In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# Cargamos las bases
df_2004 = pd.read_stata("Individual_t104.dta")
df_2024 = pd.read_excel("usu_individual_T124.xlsx")

# Ponemos los títulos en mayúsculas para filtrar la región
df_2004.columns = df_2004.columns.str.upper()
df_2024.columns = df_2024.columns.str.upper()

# Filtramos región NOA (REGION == 40)
df_2004_noa = df_2004[df_2004["REGION"] == 40]
df_2024_noa = df_2024[df_2024["REGION"] == 40]

# Unimos las bases
df_unido = pd.concat([df_2004_noa, df_2024_noa], ignore_index=True)

# Filtramos solo quienes respondieron
respondieron = df_unido[(df_unido["ESTADO"].notna()) & (df_unido["ESTADO"] != 0)]

# Creamos variable dependiente
respondieron["desocupado"] = (respondieron["ESTADO"] == 2).astype(int)

# Seleccionamos variables
variables_interes = ["CH04", "CH06", "ESTADO", "CAT_OCUP", "AGLOMERADO", "NIVEL_ED", "P47T", "P21", 
                     "CH03", "CH07", "CH08", "CH10", "CH11", "CH12", "CH13"]

# Dividimos en X e y
X = respondieron[variables_interes]
y = respondieron["desocupado"]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=444)

# Convertimos a numérico e imputamos faltantes
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test = X_test.apply(pd.to_numeric, errors='coerce').fillna(0)

# Agregamos constante
X_train = sm.add_constant(X_train, has_constant='add')
X_test = sm.add_constant(X_test, has_constant='add')

# Diferencia de medias
mean_train = X_train.mean()
mean_test = X_test.mean()
diff_means = pd.DataFrame({
    "Media Train": mean_train,
    "Media Test": mean_test,
    "Diferencia": mean_train - mean_test
})
print("Tabla de diferencia de medias entre train y test:")
print(diff_means)


In [ ]:

# PARTE B - Incisos 2, 3 y 4
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# Filtramos ocupados en entrenamiento
ocupados_train = X_train[y_train == 0].copy()
ocupados_train['salario_semanal'] = respondieron.loc[ocupados_train.index, 'P21']
ocupados_train['edad2'] = ocupados_train['CH06'] ** 2
ocupados_train['mujer'] = (ocupados_train['CH04'] == 2).astype(int)
ocupados_train['educ'] = ocupados_train['NIVEL_ED']
ocupados_train['estado_civil'] = ocupados_train['CH07']
ocupados_train['cobertura_medica'] = ocupados_train['CH08']

# Estimamos modelos
modelo1 = sm.OLS(ocupados_train['salario_semanal'], sm.add_constant(ocupados_train[['CH06']])).fit()
modelo2 = sm.OLS(ocupados_train['salario_semanal'], sm.add_constant(ocupados_train[['CH06', 'edad2']])).fit()
modelo3 = sm.OLS(ocupados_train['salario_semanal'], sm.add_constant(ocupados_train[['CH06', 'edad2', 'educ']])).fit()
modelo4 = sm.OLS(ocupados_train['salario_semanal'], sm.add_constant(ocupados_train[['CH06', 'edad2', 'educ', 'mujer']])).fit()
modelo5 = sm.OLS(ocupados_train['salario_semanal'], sm.add_constant(ocupados_train[['CH06', 'edad2', 'educ', 'mujer', 'estado_civil', 'cobertura_medica']])).fit()

def resumen_modelo(modelo):
    resumen = pd.DataFrame({
        'Coeficiente': modelo.params.round(3),
        'Desvío estándar': modelo.bse.round(2),
        'p-valor': modelo.pvalues
    })
    resumen['Significancia'] = resumen['p-valor'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.05 else '*' if p < 0.1 else ''
    )
    return resumen

for idx, mod in enumerate([modelo1, modelo2, modelo3, modelo4, modelo5], start=1):
    print(f"Modelo {idx}:
", resumen_modelo(mod), "\n")
    print(f"Modelo {idx}: N={int(mod.nobs)}, R²={mod.rsquared:.3f}\n")

# Validación en testeo
ocupados_test = X_test[y_test == 0].copy()
ocupados_test['salario_semanal'] = respondieron.loc[ocupados_test.index, 'P21']
ocupados_test['edad2'] = ocupados_test['CH06'] ** 2
ocupados_test['mujer'] = (ocupados_test['CH04'] == 2).astype(int)
ocupados_test['educ'] = ocupados_test['NIVEL_ED']
ocupados_test['estado_civil'] = ocupados_test['CH07']
ocupados_test['cobertura_medica'] = ocupados_test['CH08']

resultados = []
modelos = [
    (modelo1, ['CH06']),
    (modelo2, ['CH06', 'edad2']),
    (modelo3, ['CH06', 'edad2', 'educ']),
    (modelo4, ['CH06', 'edad2', 'educ', 'mujer']),
    (modelo5, ['CH06', 'edad2', 'educ', 'mujer', 'estado_civil', 'cobertura_medica'])
]

for idx, (modelo, variables) in enumerate(modelos, start=1):
    X_test_model = sm.add_constant(ocupados_test[variables])
    y_test_actual = ocupados_test['salario_semanal']
    y_test_pred = modelo.predict(X_test_model)
    mse = mean_squared_error(y_test_actual, y_test_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_actual, y_test_pred)
    resultados.append({'Modelo': f'Modelo {idx}', 'MSE': mse, 'RMSE': rmse, 'MAE': mae})

resultados_df = pd.DataFrame(resultados)
print("Tabla 3. Performance en testeo:")
print(resultados_df)

# Gráfico opcional
variables_modelo5 = ['CH06', 'edad2', 'educ', 'mujer', 'estado_civil', 'cobertura_medica']
X_test_model5 = sm.add_constant(ocupados_test[variables_modelo5])
salario_predicho = modelo5.predict(X_test_model5)
salario_real = ocupados_test['salario_semanal']
edad_test = ocupados_test['CH06']

plt.figure(figsize=(10, 6))
plt.scatter(edad_test, salario_real, color='blue', alpha=0.6, label='Salario Real')
plt.scatter(edad_test, salario_predicho, color='red', alpha=0.6, label='Salario Predicho', marker='x')
plt.xlabel('Edad')
plt.ylabel('Salario Semanal')
plt.title('Gráfico de Dispersión: Salario Real vs. Predicho (Modelo 5)')
plt.legend()
plt.grid(True)
plt.show()

# Nota para el informe:
# El gráfico muestra cómo se relaciona la edad con el salario semanal,
# comparando los valores reales (en azul) con las predicciones del modelo (en rojo).
# Observamos que el modelo logra capturar la tendencia general,
# aunque existen algunas desviaciones, especialmente en los extremos de edad.
# Consideramos que esto es esperable, ya que hay otros factores no incluidos en el modelo
# que también pueden influir en los ingresos.
